# Checkpoint 2 — Customer Identity Validation

## 1. Objective
The primary objective of Checkpoint 2 is to determine whether customer purchase history can be reliably constructed at the customer level using `customer_unique_id`.

Key research questions addressed:
1. What is the structural relationship between `customer_id` and `customer_unique_id`?
2. Is `customer_id` strictly an order-scoped token (1:1 with `order_id` in `orders`)?
3. How do orders in `olist_orders_dataset.csv` resolve to customer entities in `olist_customers_dataset.csv`?
4. Which order statuses represent completed/fulfilled transactions based on dataset evidence?
5. What is the actual repeat-purchase distribution across both completed purchases and all orders?
6. Is `customer_unique_id` the appropriate customer-level analytical identifier for this project?

> **Boundary Note**: This checkpoint establishes identity and repeat-purchase evidence only. No customer segmentation, RFM scoring, churn modeling, or K-Means clustering is performed.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.identity import run_identity_validation

print(f"Project root resolved: {project_root}")

## 2. Schema Validation

Execute the identity validation module to load required tables (`olist_customers_dataset.csv` and `olist_orders_dataset.csv`), verify schema presence, and verify raw data immutability.

In [ ]:
identity_data = run_identity_validation(project_root)
schema_val = identity_data["schema_validation"]

df_schema = pd.DataFrame([
    {"Dataset": "customers", "Checked Columns": ", ".join(schema_val["customers_columns_checked"]), "All Present": schema_val["customers_required_columns_present"]},
    {"Dataset": "orders", "Checked Columns": ", ".join(schema_val["orders_columns_checked"]), "All Present": schema_val["orders_required_columns_present"]}
])

print(f"Raw data immutability verified: {identity_data['raw_data_immutability_verified']}")
df_schema

## 3. Customer ID vs Customer Unique ID Analysis

Quantify mapping cardinality between transaction-scoped `customer_id` and consolidated `customer_unique_id`.

In [ ]:
cid_analysis = identity_data["customer_identifier_analysis"]

summary_stats = {
    "Metric": [
        "Total Customer Rows",
        "Unique customer_id Count",
        "Unique customer_unique_id Count",
        "Customers with Multiple customer_ids (Count)",
        "Customers with Multiple customer_ids (%)",
        "Maximum customer_ids per customer_unique_id"
    ],
    "Value": [
        f"{cid_analysis['total_customer_rows']:,}",
        f"{cid_analysis['unique_customer_id_count']:,}",
        f"{cid_analysis['unique_customer_unique_id_count']:,}",
        f"{cid_analysis['customers_with_multiple_customer_ids']:,}",
        f"{cid_analysis['customers_with_multiple_customer_ids_pct']:.2f}%",
        f"{cid_analysis['max_customer_ids_per_unique_id']}"
    ]
}
pd.DataFrame(summary_stats)

In [ ]:
dist_records = []
tot_uniq = cid_analysis["unique_customer_unique_id_count"]
for k, v in cid_analysis["customer_id_per_unique_id_distribution"].items():
    dist_records.append({
        "customer_id Values per customer_unique_id": int(k),
        "Unique Customer Count": v,
        "Percentage": round((v / tot_uniq) * 100, 4)
    })

pd.DataFrame(dist_records)

## 4. Order Cardinality & Order-to-Customer Resolution

### 4.1 Order-Customer Cardinality Validation
Explicitly validate order-level uniqueness and cardinality between `order_id` and `customer_id`.

In [ ]:
card_stats = identity_data["order_customer_cardinality_analysis"]
res_stats = identity_data["order_resolution_analysis"]

df_cardinality = pd.DataFrame([
    {"Metric": "Orders Total Rows", "Value": f"{card_stats['orders_total_rows']:,}"},
    {"Metric": "Orders Unique order_id", "Value": f"{card_stats['orders_unique_order_id']:,}"},
    {"Metric": "Orders Duplicate order_id Count", "Value": f"{card_stats['orders_duplicate_order_id_count']}"},
    {"Metric": "Orders Unique customer_id", "Value": f"{card_stats['orders_unique_customer_id']:,}"},
    {"Metric": "Customers Table Unique customer_id", "Value": f"{card_stats['customers_unique_customer_id']:,}"},
    {"Metric": "customer_id with >1 order_id in orders", "Value": f"{card_stats['customer_ids_with_multiple_orders']}"},
    {"Metric": "order_id with >1 customer_id in orders", "Value": f"{card_stats['order_ids_with_multiple_customer_ids']}"},
    {"Metric": "Strictly 1:1 order-customer cardinality", "Value": str(card_stats['is_strictly_one_to_one_order_customer'])},
    {"Metric": "Orders Resolved to customer_unique_id", "Value": f"{res_stats['orders_resolved_to_customer_id']:,} ({res_stats['resolution_percentage']:.2f}%)"}
])

print(f"Cardinality Substantiation:\n{card_stats['cardinality_substantiation']}\n")
df_cardinality

### 4.2 Order-Status Population & Delivery Evidence
Examine all order statuses and delivery-date milestone evidence to establish the completed-purchase population.

In [ ]:
status_pop = identity_data["order_status_population"]

status_rows = []
for status, count in status_pop["order_status_counts"].items():
    deliv_info = status_pop["status_delivery_date_presence"].get(status, {})
    status_rows.append({
        "Order Status": status,
        "Order Count": count,
        "Percentage": f"{status_pop['order_status_percentages'][status]:.2f}%",
        "Customer Delivery Date Present": deliv_info.get("delivered_date_present", 0),
        "Customer Delivery Date Missing": deliv_info.get("delivered_date_missing", 0)
    })

print(f"Documented Operational Definition:\n{status_pop['completed_purchase_definition']}\n")
pd.DataFrame(status_rows)

## 5. Completed-Purchase Frequency Analysis

Evaluate customer order-frequency distributions for the project's completed-purchase population (`delivered` orders), and compare side-by-side with the all-orders population.

In [ ]:
freq_data = identity_data["customer_order_frequencies"]
deliv = freq_data["completed_purchases_delivered"]
all_ord = freq_data["all_orders"]

df_freq_comp = pd.DataFrame([
    {
        "Metric": "Total Unique Customers",
        "Completed Purchases (Delivered)": f"{deliv['total_unique_customers']:,}",
        "All Orders": f"{all_ord['total_unique_customers']:,}"
    },
    {
        "Metric": "Customers with exactly 1 order",
        "Completed Purchases (Delivered)": f"{deliv['customers_with_1_order']:,} ({deliv['customers_with_1_order_pct']:.2f}%)",
        "All Orders": f"{all_ord['customers_with_1_order']:,} ({all_ord['customers_with_1_order_pct']:.2f}%)"
    },
    {
        "Metric": "Customers with exactly 2 orders",
        "Completed Purchases (Delivered)": f"{deliv['customers_with_2_orders']:,} ({deliv['customers_with_2_orders_pct']:.2f}%)",
        "All Orders": f"{all_ord['customers_with_2_orders']:,} ({all_ord['customers_with_2_orders_pct']:.2f}%)"
    },
    {
        "Metric": "Customers with 3+ orders",
        "Completed Purchases (Delivered)": f"{deliv['customers_with_3_plus_orders']:,} ({deliv['customers_with_3_plus_orders_pct']:.2f}%)",
        "All Orders": f"{all_ord['customers_with_3_plus_orders']:,} ({all_ord['customers_with_3_plus_orders_pct']:.2f}%)"
    },
    {
        "Metric": "Total Repeat Customers (2+ orders)",
        "Completed Purchases (Delivered)": f"{deliv['repeat_customers_total']:,} ({deliv['repeat_customers_pct']:.2f}%)",
        "All Orders": f"{all_ord['repeat_customers_total']:,} ({all_ord['repeat_customers_pct']:.2f}%)"
    },
    {
        "Metric": "Maximum Order Frequency",
        "Completed Purchases (Delivered)": str(deliv["max_order_frequency"]),
        "All Orders": str(all_ord["max_order_frequency"])
    }
])

df_freq_comp

In [ ]:
freq_detail_rows = []
all_keys = sorted(set(int(k) for k in deliv["frequency_distribution"].keys()) | set(int(k) for k in all_ord["frequency_distribution"].keys()))

for k in all_keys:
    c_deliv = deliv["frequency_distribution"].get(str(k), 0)
    c_all = all_ord["frequency_distribution"].get(str(k), 0)
    freq_detail_rows.append({
        "Order Count per Customer": k,
        "Delivered Customers": c_deliv,
        "Delivered %": round((c_deliv / deliv["total_unique_customers"]) * 100, 4),
        "All-Orders Customers": c_all,
        "All-Orders %": round((c_all / all_ord["total_unique_customers"]) * 100, 4)
    })

pd.DataFrame(freq_detail_rows)

## 6. Evidence-Based Conclusion

### Suitability Assessment
Based on empirical evidence:
1. **Cardinality**: `orders` exhibits exact 1:1 cardinality between `order_id` and `customer_id` (0 duplicate order IDs, 0 multi-order customer IDs, 0 multi-customer order IDs), confirming that `customer_id` is an order-scoped token.
2. **Consolidation**: `customer_unique_id` unifies multiple transaction-scoped `customer_id` records (2,997 customers with 2 to 17 orders).
3. **Resolution**: 100.0% of orders in `olist_orders_dataset.csv` resolve to `olist_customers_dataset.csv` via `customer_id`, allowing complete cross-order historical reconstruction.
4. **Repeat Purchase Visibility**: Within the project's defined delivered-order population, 2,801 of 93,358 unique customers had two or more delivered orders (3.00%), with a maximum frequency of 15 orders.

**Formal Statement**: `customer_unique_id` is the appropriate customer-level analytical identifier for this project, based on its ability to consolidate transaction-scoped `customer_id` records and support cross-order customer histories.

## 7. Checkpoint 2 Summary & Next Checkpoint

### Data Analysis Key Findings
- **Order-Customer Cardinality**: `olist_orders_dataset` has 99,441 rows, 99,441 unique `order_id`, and 99,441 unique `customer_id`. Exactly 0 `customer_id` values have multiple orders, and 0 `order_id` values have multiple customer IDs. This proves `customer_id` is strictly an order-scoped token.
- **Identifier Cardinality**: `olist_customers_dataset` contains 96,096 unique `customer_unique_id` values, consolidating 2,997 multi-order customers.
- **Join Resolution**: Exactly 99,441 of 99,441 orders (100.0%) successfully resolve from `orders.customer_id` to `customers.customer_id` and map to `customer_unique_id`.
- **Completed Purchase Population**: `delivered` status comprises 96,478 orders (97.02%), with 96,470 (99.99%) having a valid customer delivery timestamp. Canceled (625) and unavailable (609) orders represent failed/unfulfilled transactions, and remaining in-flight statuses total 1,729 orders.
- **Repeat-Purchase Rate**: Within the project's defined delivered-order population, 2,801 of 93,358 unique customers had two or more delivered orders (3.00%). Specifically, 90,557 (97.00%) have 1 order, 2,573 (2.76%) have 2 orders, and 228 (0.24%) have 3 or more orders (max = 15).
- **Raw Data Immutability**: All 9 raw CSV files retained identical SHA-256 hashes pre- and post-validation.

### Insights or Next Steps
- The ~3.00% repeat-purchase rate is an empirical constraint that will directly inform the feasibility of downstream customer segmentation and churn modeling.
- Awaiting user review and guidance before proceeding to subsequent checkpoints (such as RFM metric construction or segmentation feasibility evaluation).